# Phase 1: Macro Data Collection

## Project context

This notebook starts the Philippines Macro Nowcasting and Policy Dashboard project by creating a source inventory and collecting initial public macro data files. It does not clean, feature engineer, forecast, or build the dashboard.

## Data source strategy

- Use BSP and PSA as primary Philippines macro references because they are official domestic sources.
- Use World Bank annual indicators as backup and cross-country comparable macro context.
- Keep Phase 1 transparent: download raw files and record source status before deeper cleaning.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, OUTPUTS_DIR
from src.data_loader import (
    build_source_inventory,
    download_bsp_inflation,
    download_bsp_peso_dollar,
    fetch_world_bank_indicator,
    save_dataframe,
)

INDICATORS_DIR = OUTPUTS_DIR / "indicators"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
INDICATORS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
def summarize_dataframe(dataset, df, status="collected", notes=""):
    first_period = None
    last_period = None
    if isinstance(df, pd.DataFrame) and not df.empty:
        for col in ["year", "date", "period"]:
            if col in df.columns:
                values = df[col].dropna()
                if not values.empty:
                    first_period = values.min()
                    last_period = values.max()
                break
    return {
        "dataset": dataset,
        "rows": len(df) if isinstance(df, pd.DataFrame) else None,
        "columns": len(df.columns) if isinstance(df, pd.DataFrame) else None,
        "first_period": first_period,
        "last_period": last_period,
        "status": status,
        "notes": notes,
    }


def summarize_file(dataset, path, status="downloaded", notes=""):
    path = Path(path)
    return {
        "dataset": dataset,
        "rows": None,
        "columns": None,
        "first_period": None,
        "last_period": None,
        "status": status if path.exists() else "failed",
        "notes": notes or f"Saved raw file: {path.name}; size={path.stat().st_size if path.exists() else 0} bytes",
    }


def update_inventory_status(inventory, indicator_contains, status, notes=None):
    mask = inventory["indicator"].str.contains(indicator_contains, case=False, na=False)
    inventory.loc[mask, "status"] = status
    if notes is not None:
        inventory.loc[mask, "notes"] = notes
    return inventory


## Source inventory

In [3]:
source_inventory = build_source_inventory()
display(source_inventory)

,indicator,source,frequency,url,collection_method,status,notes
0,Inflation rate,Bangko Sentral ng Pilipinas,"Monthly or periodic, source-defined",https://www.bsp.gov.ph/statistics/excel/infrat...,Direct Excel download,planned,Primary official source for Philippines inflat...
1,USD/PHP exchange rate,Bangko Sentral ng Pilipinas,"Daily or periodic, source-defined",https://www.bsp.gov.ph/statistics/external/pes...,Direct Excel download,planned,Official BSP external statistics file for peso...
2,Key policy rates,Bangko Sentral ng Pilipinas,"Policy update, source-defined",https://www.bsp.gov.ph/SitePages/Statistics/Ke...,Manual review or later parser,manual_review_required,Page is included in inventory; automated parsi...
3,Prices and inflation reference page,Bangko Sentral ng Pilipinas,Source-defined,https://www.bsp.gov.ph/SitePages/Statistics/Pr...,Manual review or later parser,manual_review_required,Reference page for validating inflation source...
4,CPI and inflation,Philippine Statistics Authority,"Monthly, source-defined",https://psa.gov.ph/price-indices/cpi-ir,Manual review or later parser,manual_review_required,Primary national statistics reference for CPI ...
5,GDP growth annual percent,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,planned,Annual macro backup and cross-country comparab...
6,Unemployment annual percent,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,planned,Annual labor market backup indicator.
7,Inflation annual percent backup,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,planned,Annual inflation backup for cross-checking off...
8,Personal remittances received percent of GDP,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,planned,External sector and household income relevance...


## BSP inflation download

In [4]:
collection_summary = []
failed_sources = []

try:
    inflation_path = download_bsp_inflation(RAW_DATA_DIR)
    collection_summary.append(summarize_file("bsp_inflation_infrate", inflation_path))
    source_inventory = update_inventory_status(source_inventory, "Inflation rate", "downloaded")
    print(f"Downloaded BSP inflation file to {inflation_path}")
except Exception as exc:
    failed_sources.append({"dataset": "bsp_inflation_infrate", "error": f"{type(exc).__name__}: {exc}"})
    collection_summary.append(summarize_file("bsp_inflation_infrate", RAW_DATA_DIR / "bsp_inflation_infrate.xls", status="failed", notes=f"{type(exc).__name__}: {exc}"))
    source_inventory = update_inventory_status(source_inventory, "Inflation rate", "failed", f"Download failed: {type(exc).__name__}: {exc}")


Downloaded BSP inflation file to /Users/rovs/Documents/New project 2/projects/philippines-macro-nowcasting-dashboard/data/raw/bsp_inflation_infrate.xls


## BSP USD/PHP download

In [5]:
try:
    peso_dollar_path = download_bsp_peso_dollar(RAW_DATA_DIR)
    collection_summary.append(summarize_file("bsp_peso_dollar", peso_dollar_path))
    source_inventory = update_inventory_status(source_inventory, "USD/PHP", "downloaded")
    print(f"Downloaded BSP peso-dollar file to {peso_dollar_path}")
except Exception as exc:
    failed_sources.append({"dataset": "bsp_peso_dollar", "error": f"{type(exc).__name__}: {exc}"})
    collection_summary.append(summarize_file("bsp_peso_dollar", RAW_DATA_DIR / "bsp_peso_dollar.xlsx", status="failed", notes=f"{type(exc).__name__}: {exc}"))
    source_inventory = update_inventory_status(source_inventory, "USD/PHP", "failed", f"Download failed: {type(exc).__name__}: {exc}")


Downloaded BSP peso-dollar file to /Users/rovs/Documents/New project 2/projects/philippines-macro-nowcasting-dashboard/data/raw/bsp_peso_dollar.xlsx


## World Bank annual indicators download

In [6]:
world_bank_indicators = {
    "world_bank_gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "world_bank_unemployment": "SL.UEM.TOTL.ZS",
    "world_bank_inflation_backup": "FP.CPI.TOTL.ZG",
    "world_bank_remittances_pct_gdp": "BX.TRF.PWKR.DT.GD.ZS",
}

world_bank_frames = {}
for dataset, indicator_code in world_bank_indicators.items():
    try:
        df = fetch_world_bank_indicator(indicator_code, country="PHL", start_year=2000)
        output_path = RAW_DATA_DIR / f"{dataset}.csv"
        save_dataframe(df, output_path)
        world_bank_frames[dataset] = df
        collection_summary.append(summarize_dataframe(dataset, df, notes=f"Saved to {output_path.name}"))
        source_inventory.loc[source_inventory["url"].str.contains(indicator_code, regex=False), "status"] = "downloaded"
        print(f"Downloaded {dataset}: {len(df)} rows")
    except Exception as exc:
        failed_sources.append({"dataset": dataset, "error": f"{type(exc).__name__}: {exc}"})
        collection_summary.append({
            "dataset": dataset,
            "rows": 0,
            "columns": 0,
            "first_period": None,
            "last_period": None,
            "status": "failed",
            "notes": f"{type(exc).__name__}: {exc}",
        })
        source_inventory.loc[source_inventory["url"].str.contains(indicator_code, regex=False), "status"] = "failed"
        source_inventory.loc[source_inventory["url"].str.contains(indicator_code, regex=False), "notes"] = f"Download failed: {type(exc).__name__}: {exc}"


Downloaded world_bank_gdp_growth: 26 rows
Downloaded world_bank_unemployment: 26 rows
Downloaded world_bank_inflation_backup: 26 rows
Downloaded world_bank_remittances_pct_gdp: 26 rows


## Initial data availability review

In [7]:
source_inventory_path = INDICATORS_DIR / "source_inventory.csv"
collection_summary_path = INDICATORS_DIR / "data_collection_summary.csv"

source_inventory.to_csv(source_inventory_path, index=False)
collection_summary_df = pd.DataFrame(collection_summary)
collection_summary_df.to_csv(collection_summary_path, index=False)

display(source_inventory)
display(collection_summary_df)
print(f"Source inventory saved to {source_inventory_path}")
print(f"Collection summary saved to {collection_summary_path}")
failed_sources

,indicator,source,frequency,url,collection_method,status,notes
0,Inflation rate,Bangko Sentral ng Pilipinas,"Monthly or periodic, source-defined",https://www.bsp.gov.ph/statistics/excel/infrat...,Direct Excel download,downloaded,Primary official source for Philippines inflat...
1,USD/PHP exchange rate,Bangko Sentral ng Pilipinas,"Daily or periodic, source-defined",https://www.bsp.gov.ph/statistics/external/pes...,Direct Excel download,downloaded,Official BSP external statistics file for peso...
2,Key policy rates,Bangko Sentral ng Pilipinas,"Policy update, source-defined",https://www.bsp.gov.ph/SitePages/Statistics/Ke...,Manual review or later parser,manual_review_required,Page is included in inventory; automated parsi...
3,Prices and inflation reference page,Bangko Sentral ng Pilipinas,Source-defined,https://www.bsp.gov.ph/SitePages/Statistics/Pr...,Manual review or later parser,manual_review_required,Reference page for validating inflation source...
4,CPI and inflation,Philippine Statistics Authority,"Monthly, source-defined",https://psa.gov.ph/price-indices/cpi-ir,Manual review or later parser,manual_review_required,Primary national statistics reference for CPI ...
5,GDP growth annual percent,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,downloaded,Annual macro backup and cross-country comparab...
6,Unemployment annual percent,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,downloaded,Annual labor market backup indicator.
7,Inflation annual percent backup,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,downloaded,Annual inflation backup for cross-checking off...
8,Personal remittances received percent of GDP,World Bank,Annual,https://api.worldbank.org/v2/country/PHL/indic...,World Bank API,downloaded,External sector and household income relevance...


,dataset,rows,columns,first_period,last_period,status,notes
0,bsp_inflation_infrate,NaN,NaN,NaN,NaN,downloaded,Saved raw file: bsp_inflation_infrate.xls; siz...
1,bsp_peso_dollar,NaN,NaN,NaN,NaN,downloaded,Saved raw file: bsp_peso_dollar.xlsx; size=205...
2,world_bank_gdp_growth,26.0,6.0,2000.0,2025.0,collected,Saved to world_bank_gdp_growth.csv
3,world_bank_unemployment,26.0,6.0,2000.0,2025.0,collected,Saved to world_bank_unemployment.csv
4,world_bank_inflation_backup,26.0,6.0,2000.0,2025.0,collected,Saved to world_bank_inflation_backup.csv
5,world_bank_remittances_pct_gdp,26.0,6.0,2000.0,2025.0,collected,Saved to world_bank_remittances_pct_gdp.csv


Source inventory saved to /Users/rovs/Documents/New project 2/projects/philippines-macro-nowcasting-dashboard/outputs/indicators/source_inventory.csv
Collection summary saved to /Users/rovs/Documents/New project 2/projects/philippines-macro-nowcasting-dashboard/outputs/indicators/data_collection_summary.csv


[]

## Source reliability notes

- BSP and PSA are primary sources for Philippines inflation, policy, and official macro monitoring because they are domestic official institutions.
- World Bank data is useful for annual backup, cross-country comparability, and broad macro context, but it may lag local releases.
- BSP key rates and PSA CPI pages are included in the source inventory, but automated parsing is deferred until the cleaning and feature phase.

## Phase 1 limitations

- Raw files are collected but not fully cleaned.
- Excel workbooks may need manual sheet inspection before parsing.
- Forecasting target and data frequency alignment are not finalized.
- No model, nowcast, scenario analysis, or dashboard is built in this phase.

## Next steps for Phase 2 cleaning and feature engineering

- Inspect workbook sheets and standardize date fields.
- Build a clean indicator table with consistent frequencies.
- Define the first forecasting target, likely inflation-first for the MVP.
- Add lag, change, momentum, and policy-rate features after source formats are verified.